# 🌶️ Spicy Regs — Explore Public Comments in Colab

Query **24M+ federal public comments** live from the [Spicy Regs](https://github.com/civictechdc/spicy-regs) public data lake — no API key, no login, no giant downloads. Everything streams from Cloudflare R2 with DuckDB.

**What you can do here**
1. 🔎 Find a docket by keyword
2. 📊 Count comments on a docket (by month)
3. 🗂️ Browse & filter the actual comment text (search words, organizations, dates)
4. 📈 Chart comments across a set of dockets over time
5. 🏢 See the top organizations on a docket
6. 🔁 Group duplicate / near-identical comments
7. 🔥 List the busiest dockets of a year
8. ⚖️ Compare any dockets you pick

These files hold the comments regulations.gov **posts publicly**. Counts can lag the raw *received* total, since comments are posted on a delay.

▶️ **Run the Setup cell first**, then run any section. Edit the form fields on the right and re-run the cell.


In [ ]:
#@title ⚙️ Setup — run me first
# Installs DuckDB and connects to the Spicy Regs public data lake on Cloudflare R2.
!pip -q install duckdb pandas matplotlib

import duckdb, pandas as pd
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.max_rows", 100)

R2 = "https://r2.spicy-regs.dev"
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")

DOCKETS = f"read_parquet('{R2}/dockets.parquet')"
INDEX   = f"read_parquet('{R2}/comments_index.parquet')"

# Cap how many partition files a single comment query will scan.
# Each partition = one HTTP request; broad scans can trip R2's rate limiter (HTTP 429).
MAX_PARTITIONS = 40

def comment_partitions(docket_id):
    """One row per (year, month) partition that exists for this docket, with row_count."""
    return con.execute(f"""
        SELECT agency_code, docket_id, year, month, row_count
        FROM {INDEX} WHERE docket_id = ? ORDER BY year, month
    """, [docket_id]).fetchdf()

def comments_source(parts):
    """Build a DuckDB read_parquet(...) over just this docket's partition files.
    union_by_name reconciles older partitions that have fewer columns."""
    urls = [f"{R2}/comments/agency_code={r.agency_code}/docket_id={r.docket_id}"
            f"/year={int(r.year)}/month={int(r.month)}/part-0.parquet"
            for r in parts.itertuples()]
    return f"read_parquet({urls}, union_by_name=true)", len(urls)

print("✅ ready — connected to", R2)



## 🔎 1. Find a docket by keyword

Search docket titles + abstracts. Grab the `docket_id` you want and paste it into the sections below.


In [ ]:
#@title 🔎 Find a docket
keyword     = "financial assistance"  #@param {type:"string"}
agency_code = ""                       #@param {type:"string"}
max_results = 20                        #@param {type:"integer"}

where  = ["(title ILIKE '%'||?||'%' OR abstract ILIKE '%'||?||'%')"]
params = [keyword, keyword]
if agency_code.strip():
    where.append("agency_code = ?"); params.append(agency_code.strip())

df = con.execute(f"""
    SELECT docket_id, agency_code, docket_type, modify_date, title
    FROM {DOCKETS}
    WHERE {' AND '.join(where)}
    ORDER BY modify_date DESC
    LIMIT {int(max_results)}
""", params).fetchdf()

print(f"{len(df)} docket(s) matching '{keyword}'" + (f" in {agency_code}" if agency_code.strip() else ""))
df



## 📊 2. How many comments on a docket?

Fast count straight from the tiny index file (no comment scan). Also breaks it down by month.


In [ ]:
#@title 📊 Count comments
docket_id = "OMB-2026-0034"  #@param {type:"string"}

parts = comment_partitions(docket_id)
if parts.empty:
    print(f"⚠️ No comments found for {docket_id}. Double-check the ID with section 1.")
else:
    total  = int(parts.row_count.sum())
    agency = parts.agency_code.iloc[0]
    print(f"📨 {total:,} posted comments on {docket_id}  ({agency})")
    by_month = parts.assign(
        period = parts.year.astype(int).astype(str) + "-" + parts.month.astype(int).astype(str).str.zfill(2)
    )[["period", "row_count"]].rename(columns={"row_count": "comments"})
    display(by_month)
    ax = by_month.set_index("period")["comments"].plot(
        kind="bar", figsize=(10, 3), title=f"Comments per month — {docket_id}")
    ax.set_xlabel(""); ax.figure.tight_layout()



## 🗂️ 3. Browse & filter the actual comments

Search inside the comment bodies and filter by organization / date. Leave a field blank to skip that filter.
Dates are `YYYY-MM-DD`. Results stream from just this docket's partition files.


In [ ]:
#@title 🗂️ Browse & filter comments
docket_id     = "OMB-2026-0034"  #@param {type:"string"}
text_contains = ""                #@param {type:"string"}
org_contains  = ""                #@param {type:"string"}
posted_after  = ""                #@param {type:"string"}
posted_before = ""                #@param {type:"string"}
limit         = 25                #@param {type:"integer"}

parts = comment_partitions(docket_id)
if parts.empty:
    print(f"⚠️ No comments found for {docket_id}. Check the ID with section 1.")
else:
    if len(parts) > MAX_PARTITIONS:
        print(f"⚠️ {docket_id} spans {len(parts)} partitions; scanning the {MAX_PARTITIONS} "
              f"largest to avoid rate limits. Narrow by date for full coverage.")
        parts = parts.sort_values("row_count", ascending=False).head(MAX_PARTITIONS)
    src, n_urls = comments_source(parts)

    filt = ["1=1"]; params = []
    if text_contains.strip():
        filt.append("(comment ILIKE '%'||?||'%' OR title ILIKE '%'||?||'%' "
                    "OR coalesce(text_content,'') ILIKE '%'||?||'%')")
        params += [text_contains, text_contains, text_contains]
    if org_contains.strip():
        filt.append("organization ILIKE '%'||?||'%'"); params.append(org_contains)
    if posted_after.strip():
        filt.append("left(posted_date, 10) >= ?"); params.append(posted_after)
    if posted_before.strip():
        filt.append("left(posted_date, 10) <= ?"); params.append(posted_before)

    where_sql = " AND ".join(filt)
    matched = con.execute(f"SELECT COUNT(*) FROM {src} WHERE {where_sql}", params).fetchone()[0]

    df = con.execute(f"""
        SELECT comment_id, posted_date, first_name, last_name, organization, category,
               left(coalesce(nullif(comment,''), text_content, ''), 400) AS comment_preview
        FROM {src}
        WHERE {where_sql}
        ORDER BY posted_date DESC
        LIMIT {int(limit)}
    """, params).fetchdf()

    print(f"💬 {matched:,} comment(s) match — showing {len(df)} "
          f"(scanned {n_urls} partition file(s))")
    display(df)



## 📈 4. Chart comments across a set of dockets

Example: the four 2 CFR / OMB grant-guidance dockets across 13 years. Swap in any docket IDs you like.


In [ ]:
#@title 📈 Comments by docket
targets = {
    "2013": "OMB-2013-0001",   # Reform of Federal Policies Relating to Grants (Uniform Guidance)
    "2019": "OMB-2019-0005",   # Guidance for Grants and Agreements
    "2023": "OMB-2023-0004",   # 2 CFR Proposed Guidance
    "2026": "OMB-2026-0034",   # Regulation for Federal Financial Assistance
}
rows = []
for label, dk in targets.items():
    n = con.execute(f"SELECT COALESCE(SUM(row_count),0) FROM {INDEX} WHERE docket_id = ?", [dk]).fetchone()[0]
    rows.append({"label": label, "docket": dk, "posted_comments": int(n)})

chart = pd.DataFrame(rows).set_index("label")
display(chart)
ax = chart["posted_comments"].plot(kind="bar", figsize=(8, 4), title="Posted public comments by docket")
ax.set_xlabel(""); ax.bar_label(ax.containers[0], fmt="{:,.0f}"); ax.figure.tight_layout()



## 🏢 5. Top organizations on a docket

Which organizations filed the most comments. (Many comments have no organization — those are dropped here.)


In [ ]:
#@title 🏢 Top organizations
docket_id = "OMB-2026-0034"  #@param {type:"string"}
top_n     = 20                #@param {type:"integer"}

parts = comment_partitions(docket_id)
if parts.empty:
    print(f"⚠️ No comments found for {docket_id}.")
else:
    if len(parts) > MAX_PARTITIONS:
        print(f"⚠️ {docket_id} spans {len(parts)} partitions; counting only the {MAX_PARTITIONS} largest, so the numbers below are a floor. Use section 2 for the full total.")
        parts = parts.sort_values("row_count", ascending=False).head(MAX_PARTITIONS)
    src, _ = comments_source(parts)
    df = con.execute(f"""
        SELECT organization, COUNT(*) AS comments
        FROM {src}
        WHERE organization IS NOT NULL AND trim(organization) <> ''
        GROUP BY organization ORDER BY comments DESC LIMIT {int(top_n)}
    """).fetchdf()
    print(f"Top {len(df)} organizations by comment count on {docket_id}")
    display(df)



## 🔁 6. Find duplicate / near-identical comments

A utility for grouping comments by identical body text — handy for seeing how much of a docket is repeated text vs. unique submissions. For `OMB-2026-0034` the posted comments are mostly distinct (~1.1× duplication), so this is a general tool, not evidence about any particular docket.


In [ ]:
#@title 🔁 Group identical comment bodies
docket_id = "OMB-2026-0034"  #@param {type:"string"}
top_n     = 10                #@param {type:"integer"}

parts = comment_partitions(docket_id)
if parts.empty:
    print(f"⚠️ No comments found for {docket_id}.")
else:
    if len(parts) > MAX_PARTITIONS:
        print(f"⚠️ {docket_id} spans {len(parts)} partitions; counting only the {MAX_PARTITIONS} largest, so the numbers below are a floor. Use section 2 for the full total.")
        parts = parts.sort_values("row_count", ascending=False).head(MAX_PARTITIONS)
    src, _ = comments_source(parts)
    total, uniq = con.execute(f"""
        SELECT COUNT(*),
               COUNT(DISTINCT trim(lower(coalesce(nullif(comment,''), text_content, ''))))
        FROM {src}
    """).fetchone()
    ratio = total / max(uniq, 1)
    print(f"📊 {total:,} posted comments  •  {uniq:,} unique bodies  •  {ratio:.1f}× duplication\n")
    df = con.execute(f"""
        WITH c AS (SELECT trim(coalesce(nullif(comment,''), text_content, '')) AS body FROM {src})
        SELECT COUNT(*) AS copies, left(body, 300) AS comment_text
        FROM c WHERE length(body) > 40
        GROUP BY body ORDER BY copies DESC LIMIT {int(top_n)}
    """).fetchdf()
    display(df)



## 🔥 7. Busiest dockets of a year (all agencies)

The most-commented dockets across the whole government in a given year — straight from the index, no comment scan.


In [ ]:
#@title 🔥 Busiest dockets of a year
year  = 2026  #@param {type:"integer"}
top_n = 20    #@param {type:"integer"}

df = con.execute(f"""
    SELECT docket_id, agency_code, SUM(row_count) AS comments
    FROM {INDEX} WHERE year = {int(year)}
    GROUP BY docket_id, agency_code ORDER BY comments DESC LIMIT {int(top_n)}
""").fetchdf()
print(f"Most-commented dockets posted in {year}")
display(df)



## ⚖️ 8. Compare any set of dockets

Paste any comma-separated docket IDs to chart their posted-comment counts side by side.


In [ ]:
#@title ⚖️ Compare dockets
dockets = "OMB-2013-0001, OMB-2019-0005, OMB-2023-0004, OMB-2026-0034"  #@param {type:"string"}

ids = [d.strip() for d in dockets.split(",") if d.strip()]
rows = []
for dk in ids:
    n = con.execute(f"SELECT COALESCE(SUM(row_count),0) FROM {INDEX} WHERE docket_id = ?", [dk]).fetchone()[0]
    rows.append({"docket": dk, "posted_comments": int(n)})

comp = pd.DataFrame(rows).set_index("docket")
display(comp)
ax = comp["posted_comments"].plot(kind="barh", figsize=(8, 0.6*len(ids)+1),
        title="Posted comments by docket")
ax.set_ylabel(""); ax.bar_label(ax.containers[0], fmt="{:,.0f}"); ax.figure.tight_layout()

